In [3]:
# shopify_test_sku.py
import os
import sys
import json
import requests

SHOP =  "71eaf7.myshopify.com"
ADMIN_ACCESS_TOKEN = os.getenv("SHOPIFY_ADMIN_ACCESS_TOKEN")
API_VERSION = "2025-10"

def gql(query: str, variables: dict):
    if not SHOP or not ADMIN_ACCESS_TOKEN:
        raise RuntimeError("Please set env SHOPIFY_SHOP and SHOPIFY_ADMIN_ACCESS_TOKEN")

    url = f"https://{SHOP}/admin/api/{API_VERSION}/graphql.json"
    headers = {
        "Content-Type": "application/json",
        "X-Shopify-Access-Token": ADMIN_ACCESS_TOKEN,
    }
    r = requests.post(url, headers=headers, json={"query": query, "variables": variables}, timeout=30)
    r.raise_for_status()
    data = r.json()
    if "errors" in data:
        raise RuntimeError(f"GraphQL errors: {json.dumps(data['errors'], ensure_ascii=False)}")
    return data["data"]

QUERY_BY_SKU = """
query($q: String!) {
  productVariants(first: 5, query: $q) {
    edges {
      node {
        id
        sku
        title
        product {
          id
          title
          handle
          status
          createdAt
          updatedAt
        }
        inventoryItem { id }
      }
    }
  }
}
"""

MUTATION_PRODUCT_TITLE_UPDATE = """
mutation($input: ProductInput!) {
  productUpdate(input: $input) {
    product { id title handle status updatedAt }
    userErrors { field message }
  }
}
"""

MUTATION_VARIANT_TITLE_UPDATE = """
mutation($input: ProductVariantInput!) {
  productVariantUpdate(input: $input) {
    productVariant { id title sku }
    userErrors { field message }
  }
}
"""

def find_by_sku(sku: str):
    q = f"sku:{sku}"
    data = gql(QUERY_BY_SKU, {"q": q})
    edges = data["productVariants"]["edges"]
    return [e["node"] for e in edges]

def update_product_title(product_id: str, new_title: str):
    data = gql(MUTATION_PRODUCT_TITLE_UPDATE, {"input": {"id": product_id, "title": new_title}})
    res = data["productUpdate"]
    if res["userErrors"]:
        raise RuntimeError(f"userErrors: {res['userErrors']}")
    return res["product"]

def update_variant_title(variant_id: str, new_title: str):
    data = gql(MUTATION_VARIANT_TITLE_UPDATE, {"input": {"id": variant_id, "title": new_title}})
    res = data["productVariantUpdate"]
    if res["userErrors"]:
        raise RuntimeError(f"userErrors: {res['userErrors']}")
    return res["productVariant"]

def main():
    if len(sys.argv) < 2:
        print("Usage:")
        print("  python shopify_test_sku.py <SKU> [--set-product-title 'New Title'] [--set-variant-title 'New Variant Title']")
        sys.exit(1)

    sku = sys.argv[1]
    nodes = find_by_sku(sku)

    print(f"Found {len(nodes)} variants for SKU={sku}")
    for i, n in enumerate(nodes):
        print(f"\n[{i}] variant_id={n['id']}")
        print(f"    sku={n['sku']}")
        print(f"    variant_title={n['title']}")
        print(f"    product_id={n['product']['id']}")
        print(f"    product_title={n['product']['title']}")
        print(f"    status={n['product']['status']}")
        print(f"    handle={n['product']['handle']}")
        print(f"    createdAt={n['product']['createdAt']}")
        print(f"    updatedAt={n['product']['updatedAt']}")
        print(f"    inventory_item_id={n['inventoryItem']['id']}")

    # Optional updates:
    if "--set-product-title" in sys.argv:
        idx = sys.argv.index("--set-product-title")
        new_title = sys.argv[idx + 1]
        if not nodes:
            raise RuntimeError("No variant found; cannot update product title.")
        prod = update_product_title(nodes[0]["product"]["id"], new_title)
        print("\n✅ Product title updated:", prod)

    if "--set-variant-title" in sys.argv:
        idx = sys.argv.index("--set-variant-title")
        new_title = sys.argv[idx + 1]
        if not nodes:
            raise RuntimeError("No variant found; cannot update variant title.")
        v = update_variant_title(nodes[0]["id"], new_title)
        print("\n✅ Variant title updated:", v)

if __name__ == "__main__":
    main()


Found 0 variants for SKU=--f=c:\Users\mingc\AppData\Roaming\jupyter\runtime\kernel-v33df0893d2dd06e72e77792e1cd50a49eca934562.json


In [12]:
import os
import json
import requests
from typing import Any, Dict, List

SHOP =  "71eaf7.myshopify.com"
TOKEN = os.getenv("SHOPIFY_ADMIN_ACCESS_TOKEN")
API_VERSION = "2025-10"

def gql(query: str, variables: dict) -> dict:
    if not SHOP or not TOKEN:
        raise RuntimeError("Please set env: SHOPIFY_SHOP and SHOPIFY_ADMIN_ACCESS_TOKEN")

    url = f"https://{SHOP}/admin/api/{API_VERSION}/graphql.json"
    headers = {
        "Content-Type": "application/json",
        "X-Shopify-Access-Token": TOKEN,
    }
    r = requests.post(url, headers=headers, json={"query": query, "variables": variables}, timeout=45)
    r.raise_for_status()
    payload = r.json()
    if "errors" in payload:
        raise RuntimeError("GraphQL errors:\n" + json.dumps(payload["errors"], ensure_ascii=False, indent=2))
    return payload["data"]


QUERY_MIN_BY_SKU = """
query MinBySku($q: String!) {
  productVariants(first: 5, query: $q) {
    edges {
      node {
        id
        sku
        price
        compareAtPrice
        availableForSale
        inventoryPolicy
        inventoryQuantity

        product {
          id
          title
          description
          descriptionHtml

          featuredImage { url altText width height }

          images(first: 100) {
            edges {
              node { url altText width height }
            }
          }

          media(first: 50) {
            edges {
              node {
                __typename
                ... on MediaImage {
                  image { url altText width height }
                }
                ... on ExternalVideo {
                  embeddedUrl
                  host
                }
                ... on Video {
                  sources { url mimeType format height width }
                }
                ... on Model3d {
                  sources { url mimeType format }
                }
              }
            }
          }
        }

        inventoryItem {
          id
          tracked
          inventoryLevels(first: 250) {
            edges {
              node {
                location { id name isActive }
                quantities(names: ["available", "on_hand", "committed", "incoming"]) {
                  name
                  quantity
                }
              }
            }
          }
        }
      }
    }
  }
}
"""


def normalize_media(product: Dict[str, Any]) -> Dict[str, Any]:
    # 统一把图片/视频等输出成一个 media 列表，方便你存库
    media_items: List[Dict[str, Any]] = []

    # images（几乎所有店都有）
    for e in (product.get("images") or {}).get("edges", []):
        img = e["node"]
        media_items.append({
            "type": "image",
            "url": img.get("url"),
            "alt": img.get("altText"),
            "width": img.get("width"),
            "height": img.get("height"),
        })

    # media（更泛化：可能包含视频/3d等）
    for e in (product.get("media") or {}).get("edges", []):
        node = e["node"]
        t = node.get("__typename")

        if t == "MediaImage" and node.get("image"):
            img = node["image"]
            media_items.append({
                "type": "image",
                "url": img.get("url"),
                "alt": img.get("altText"),
                "width": img.get("width"),
                "height": img.get("height"),
                "source": "media",
            })
        elif t == "ExternalVideo":
            media_items.append({
                "type": "external_video",
                "url": node.get("embeddedUrl"),
                "host": node.get("host"),
                "source": "media",
            })
        elif t == "Video":
            for s in node.get("sources", []) or []:
                media_items.append({
                    "type": "video",
                    "url": s.get("url"),
                    "mimeType": s.get("mimeType"),
                    "format": s.get("format"),
                    "height": s.get("height"),
                    "width": s.get("width"),
                    "source": "media",
                })
        elif t == "Model3d":
            for s in node.get("sources", []) or []:
                media_items.append({
                    "type": "model3d",
                    "url": s.get("url"),
                    "mimeType": s.get("mimeType"),
                    "format": s.get("format"),
                    "source": "media",
                })
        else:
            media_items.append({"type": "unknown", "raw": node})

    return {
        "featured_image": product.get("featuredImage"),
        "media": media_items,
    }


def normalize_inventory(inv_item: Dict[str, Any]) -> Dict[str, Any]:
    levels = []
    for e in (inv_item.get("inventoryLevels") or {}).get("edges", []):
        node = e["node"]
        loc = node.get("location") or {}
        qty_map = {q["name"]: q["quantity"] for q in (node.get("quantities") or [])}
        levels.append({
            "location_id": loc.get("id"),
            "location_name": loc.get("name"),
            "location_active": loc.get("isActive"),
            "quantities": qty_map,  # available/on_hand/committed/incoming
        })

    # 常用：把所有 location 的 available 也求个总和，方便你业务判断 sold out
    total_available = sum((lvl["quantities"].get("available") or 0) for lvl in levels)

    return {
        "tracked": inv_item.get("tracked"),
        "total_available": total_available,
        "levels": levels,
    }


def main():
    sku = os.getenv("SKU", "AK3CVN6")
    data = gql(QUERY_MIN_BY_SKU, {"q": f"sku:{sku}"})

    edges = data["productVariants"]["edges"]
    if not edges:
        print(json.dumps({"ok": False, "error": "SKU not found", "sku": sku}, ensure_ascii=False, indent=2))
        return

    # 你是单件 SKU，正常只会匹配 1 个；这里如果匹配多个也全输出
    out_variants = []
    for e in edges:
        v = e["node"]
        p = v["product"]
        inv = v["inventoryItem"]

        media_norm = normalize_media(p)
        inv_norm = normalize_inventory(inv)

        out_variants.append({
            "variant_id": v["id"],
            "product_id": p["id"],
            "sku": v.get("sku"),
            "title": p.get("title"),
            "description": p.get("description"),          # 纯文本
            "description_html": p.get("descriptionHtml"), # 如果你要保留富文本
            "price": v.get("price"),
            "compare_at_price": v.get("compareAtPrice"),
            "available_for_sale": v.get("availableForSale"),
            "inventory_policy": v.get("inventoryPolicy"),
            "inventory_quantity": v.get("inventoryQuantity"),  # Shopify 的一个汇总数（不一定等同各 location）
            "media": media_norm,
            "inventory": inv_norm,
        })

    result = {
        "ok": True,
        "sku_query": sku,
        "matches": len(out_variants),
        "items": out_variants,
    }

    print(json.dumps(result, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


{
  "ok": true,
  "sku_query": "AK3CVN6",
  "matches": 1,
  "items": [
    {
      "variant_id": "gid://shopify/ProductVariant/48440014045408",
      "product_id": "gid://shopify/Product/9107475398880",
      "sku": "AK3CVN6",
      "title": "Chanel Diana Reissue Small - 22CM Quilted Lambskin Red Gold Hardware Series 20",
      "description": "Condition · Item in overall excellent condition. All our items are pre-loved unless stated brand new. · We strive to capture all details and highlight any defects. Please closely inspect detailed photos or request further images/videos before purchase. Measurements 22 x 15.2 x 7.6cm Package includes · Authenticity card (Series 20) · Hologram sticker · Dust bag and box · Entrupy Certificate (Upon request for extra $50 SGD, please contact us for this add-on service)",
      "description_html": "<h4>Condition</h4>\n<hr>\n<p>· Item in overall excellent condition. All our items are pre-loved unless stated brand new. <br>· We strive to capture all deta

In [13]:
import os
import json
import requests
from typing import Any, Dict, List

SHOP =  "71eaf7.myshopify.com"
TOKEN = os.getenv("SHOPIFY_ADMIN_ACCESS_TOKEN")
API_VERSION = "2025-10"

def gql(query: str, variables: dict) -> dict:
    if not SHOP or not TOKEN:
        raise RuntimeError("Please set env: SHOPIFY_SHOP and SHOPIFY_ADMIN_ACCESS_TOKEN")

    url = f"https://{SHOP}/admin/api/{API_VERSION}/graphql.json"
    headers = {
        "Content-Type": "application/json",
        "X-Shopify-Access-Token": TOKEN,
    }
    r = requests.post(url, headers=headers, json={"query": query, "variables": variables}, timeout=45)
    r.raise_for_status()
    payload = r.json()
    if "errors" in payload:
        raise RuntimeError("GraphQL errors:\n" + json.dumps(payload["errors"], ensure_ascii=False, indent=2))
    return payload["data"]


QUERY_MIN_BY_SKU = """
query MinBySku($q: String!) {
  productVariants(first: 5, query: $q) {
    edges {
      node {
        id
        sku
        price
        compareAtPrice
        availableForSale
        inventoryPolicy
        inventoryQuantity

        product {
          id
          title
          description
          descriptionHtml

          featuredImage { url altText width height }

          images(first: 100) {
            edges {
              node { url altText width height }
            }
          }

          media(first: 50) {
            edges {
              node {
                __typename
                ... on MediaImage {
                  image { url altText width height }
                }
                ... on ExternalVideo {
                  embeddedUrl
                  host
                }
                ... on Video {
                  sources { url mimeType format height width }
                }
                ... on Model3d {
                  sources { url mimeType format }
                }
              }
            }
          }
        }

        inventoryItem {
          id
          tracked
          inventoryLevels(first: 250) {
            edges {
              node {
                location { id name isActive }
                quantities(names: ["available", "on_hand", "committed", "incoming"]) {
                  name
                  quantity
                }
              }
            }
          }
        }
      }
    }
  }
}
"""


def normalize_media(product: Dict[str, Any]) -> Dict[str, Any]:
    # 统一把图片/视频等输出成一个 media 列表，方便你存库
    media_items: List[Dict[str, Any]] = []

    # images（几乎所有店都有）
    for e in (product.get("images") or {}).get("edges", []):
        img = e["node"]
        media_items.append({
            "type": "image",
            "url": img.get("url"),
            "alt": img.get("altText"),
            "width": img.get("width"),
            "height": img.get("height"),
        })

    # media（更泛化：可能包含视频/3d等）
    for e in (product.get("media") or {}).get("edges", []):
        node = e["node"]
        t = node.get("__typename")

        if t == "MediaImage" and node.get("image"):
            img = node["image"]
            media_items.append({
                "type": "image",
                "url": img.get("url"),
                "alt": img.get("altText"),
                "width": img.get("width"),
                "height": img.get("height"),
                "source": "media",
            })
        elif t == "ExternalVideo":
            media_items.append({
                "type": "external_video",
                "url": node.get("embeddedUrl"),
                "host": node.get("host"),
                "source": "media",
            })
        elif t == "Video":
            for s in node.get("sources", []) or []:
                media_items.append({
                    "type": "video",
                    "url": s.get("url"),
                    "mimeType": s.get("mimeType"),
                    "format": s.get("format"),
                    "height": s.get("height"),
                    "width": s.get("width"),
                    "source": "media",
                })
        elif t == "Model3d":
            for s in node.get("sources", []) or []:
                media_items.append({
                    "type": "model3d",
                    "url": s.get("url"),
                    "mimeType": s.get("mimeType"),
                    "format": s.get("format"),
                    "source": "media",
                })
        else:
            media_items.append({"type": "unknown", "raw": node})

    return {
        "featured_image": product.get("featuredImage"),
        "media": media_items,
    }


def normalize_inventory(inv_item: Dict[str, Any]) -> Dict[str, Any]:
    levels = []
    for e in (inv_item.get("inventoryLevels") or {}).get("edges", []):
        node = e["node"]
        loc = node.get("location") or {}
        qty_map = {q["name"]: q["quantity"] for q in (node.get("quantities") or [])}
        levels.append({
            "location_id": loc.get("id"),
            "location_name": loc.get("name"),
            "location_active": loc.get("isActive"),
            "quantities": qty_map,  # available/on_hand/committed/incoming
        })

    # 常用：把所有 location 的 available 也求个总和，方便你业务判断 sold out
    total_available = sum((lvl["quantities"].get("available") or 0) for lvl in levels)

    return {
        "tracked": inv_item.get("tracked"),
        "total_available": total_available,
        "levels": levels,
    }


def main():
    sku = os.getenv("SKU", "AK3CVN7")
    data = gql(QUERY_MIN_BY_SKU, {"q": f"sku:{sku}"})

    edges = data["productVariants"]["edges"]
    if not edges:
        print(json.dumps({"ok": False, "error": "SKU not found", "sku": sku}, ensure_ascii=False, indent=2))
        return

    # 你是单件 SKU，正常只会匹配 1 个；这里如果匹配多个也全输出
    out_variants = []
    for e in edges:
        v = e["node"]
        p = v["product"]
        inv = v["inventoryItem"]

        media_norm = normalize_media(p)
        inv_norm = normalize_inventory(inv)

        out_variants.append({
            "variant_id": v["id"],
            "product_id": p["id"],
            "sku": v.get("sku"),
            "title": p.get("title"),
            "description": p.get("description"),          # 纯文本
            "description_html": p.get("descriptionHtml"), # 如果你要保留富文本
            "price": v.get("price"),
            "compare_at_price": v.get("compareAtPrice"),
            "available_for_sale": v.get("availableForSale"),
            "inventory_policy": v.get("inventoryPolicy"),
            "inventory_quantity": v.get("inventoryQuantity"),  # Shopify 的一个汇总数（不一定等同各 location）
            "media": media_norm,
            "inventory": inv_norm,
        })

    result = {
        "ok": True,
        "sku_query": sku,
        "matches": len(out_variants),
        "items": out_variants,
    }

    print(json.dumps(result, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


{
  "ok": false,
  "error": "SKU not found",
  "sku": "AK3CVN7"
}
